# Indic-FinSight: Multi-Agent Financial Analyst
**Kaggle Submission — Gemma 2B on T4 GPU**

A multi-agent system that breaks complex financial queries into sub-tasks,
delegates to specialized agents (Filings, Market, Chart, Web), and synthesizes
a unified answer with follow-up options.

**Architecture:**
```
User Query → Intent Router → [Filings | Market | Chart | Web] → Orchestrator → Response
```


In [ ]:
!pip install -q -U chromadb fastembed yfinance transformers accelerate bitsandbytes fastapi uvicorn pydantic nest-asyncio duckduckgo-search
!npm install -g localtunnel

In [ ]:
import torch
import chromadb
import yfinance as yf
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from duckduckgo_search import DDGS
import re
import json
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# 1. Financial Document Ingestion (ChromaDB Vector Store)

dense_financial_text = '''
Reliance Industries Q3 FY26 Earnings Call Transcript Excerpt:
The management recognizes that the global semiconductor shortage continues to pose a supply chain risk, though we expect it to stabilize by Q4.
Commodity inflation, particularly in steel and lithium, has impacted our margin by 120 basis points.
On a positive note, our Retail segment and Jio saw a 45% YoY growth.
We remain committed to our net-zero carbon emission goal by 2040.
Capital expenditure for the upcoming fiscal is slated at 8,000 Crores, largely directed towards 5G expansion.
Revenue for Q3 FY26 stood at Rs 2,40,000 Crores, up 12% YoY.
Net profit for Q3 FY26 was Rs 18,500 Crores, a 15% increase over the previous year.
The O2C segment reported EBITDA of Rs 15,200 Crores with margins at 8.2%.
Jio Platforms added 12 million subscribers this quarter, taking total to 482 million.
Retail segment revenue crossed Rs 75,000 Crores with 3,200 new store openings.
Debt-to-equity ratio improved to 0.32 from 0.38 in the previous quarter.
The company announced a dividend of Rs 10 per share for FY26.
'''

print("Initializing ChromaDB vector store...")
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="financial_filings")

chunks = [chunk.strip() for chunk in dense_financial_text.split('.') if len(chunk.strip()) > 20]

collection.add(
    documents=chunks,
    metadatas=[{"source": "Reliance Industries Q3 FY26 Transcript"} for _ in chunks],
    ids=[str(i) for i in range(len(chunks))]
)
print(f"Ingested {len(chunks)} chunks into vector store.")

In [ ]:
# 2. Load Gemma 2B from Kaggle Model Hub
import os
import glob

print("Locating Gemma 2 model weights...")
possible_paths = glob.glob("/kaggle/input/**/config.json", recursive=True)

model_path = None
for p in possible_paths:
    if "gemma" in p.lower():
        model_path = os.path.dirname(p)
        break

if not model_path:
    raise FileNotFoundError("Gemma model not attached. Add it via Kaggle sidebar.")

loaded_model_name = [part for part in model_path.split('/') if "gemma" in part.lower()][-1]
print(f"Loading {loaded_model_name} from: {model_path}")

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype=torch.float16,
)

gemma = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.1,
)
print("Model loaded.")

In [ ]:
# 3. Tool Definitions

def search_filings(query: str) -> str:
    results = collection.query(query_texts=[query], n_results=3)
    return " ".join(results['documents'][0])

def get_live_stock_price(ticker: str) -> str:
    try:
        stock = yf.Ticker(ticker.strip())
        hist = stock.history(period="5d")
        if hist.empty:
            return f"No data found for {ticker}."
        price = hist['Close'].iloc[-1]
        prev = hist['Close'].iloc[-2] if len(hist) > 1 else price
        change = ((price - prev) / prev) * 100
        direction = "up" if change >= 0 else "down"
        return f"{ticker}: ₹{price:.2f} ({direction} {abs(change):.2f}% from previous close)"
    except Exception as e:
        return f"Error fetching {ticker}: {e}"

def search_web(query: str) -> str:
    try:
        # DDGS news endpoint frequently 403s, text is more stable
        results = DDGS().text(query, max_results=3)
        if not results:
            return "No web results found."
        return " ".join([f"[{r['title']}] {r['body']}" for r in results])
    except Exception as e:
        return f"Web search currently unavailable due to rate limits. Instruct the user to try again later."

def plot_chart(type_title_data: str) -> str:
    return f"[CHART:{type_title_data}]"

TOOLS = {
    "search_filings": search_filings,
    "get_live_stock_price": get_live_stock_price,
    "search_web": search_web,
    "plot_chart": plot_chart,
}

In [ ]:
# 4. Multi-Agent Orchestration System

INTENT_KEYWORDS = {
    "filings": ["risk", "risks", "filing", "report", "earnings", "revenue", "profit",
                "margin", "capex", "growth", "debt", "ebitda", "dividend", "subscriber",
                "supply chain", "segment", "quarter", "annual", "fy", "q1", "q2", "q3", "q4"],
    "market":  ["price", "stock", "ticker", "nse", "bse", "live", "current", "share price",
                "market cap", "trading"],
    "chart":   ["chart", "graph", "plot", "visualize", "bar chart", "trend", "compare", "pie", "line"],
    "web":     ["news", "recent", "search", "update", "latest", "internet", "web"],
}

def classify_intent(query: str) -> list:
    query_lower = query.lower()
    intents = []
    for intent, keywords in INTENT_KEYWORDS.items():
        if any(kw in query_lower for kw in keywords):
            intents.append(intent)
    if not intents:
        intents = ["web"]  # Default to web search if unknown
    return intents

AGENT_PROMPTS = {
    "filings": '''You are FilingsAgent, a financial document analyst.
You have ONE tool: search_filings(query)
Your job: search the company filings database and extract relevant information.

CRITICAL INSTRUCTIONS:
- You CANNOT use any other tools. If you need something else, answer based on your knowledge.
- When extracting data for a chart or graph, search ONLY for the underlying entities or metrics (e.g., "Reliance segment revenues"). DO NOT include words like "chart", "graph", or "plot" in your search query!

Format your response EXACTLY as:
Thought: I need to search the filings for this information.
Action: search_filings
Action Input: [your search query]

After receiving an Observation, provide:
Thought: [analysis of what you found]
Result: [concise summary of findings]''',

    "market": '''You are MarketAgent, a live market data specialist.
You have ONE tool: get_live_stock_price(ticker)
Your job: fetch the current stock price for the requested ticker.

CRITICAL INSTRUCTIONS:
- NEVER HALLUCINATE TOOL NAMES. YOU CAN ONLY USE `get_live_stock_price`. DO NOT use `search_news` or any other tool.
- Always append .NS to Indian stock tickers (e.g. RELIANCE.NS, TCS.NS)

Format your response EXACTLY as:
Thought: I need to get the stock price.
Action: get_live_stock_price
Action Input: [TICKER.NS format]

After receiving an Observation, provide:
Thought: [brief analysis]
Result: [the price data]''',

    "web": '''You are WebAgent, a live internet search specialist.
You have ONE tool: search_web(query)
Your job: search the internet for the most recent news or financial information.

CRITICAL INSTRUCTIONS:
- NEVER HALLUCINATE TOOL NAMES. YOU CAN ONLY USE `search_web`. DO NOT use `search_news`, `search_internet`, or any other tool.
- Keep your search queries short and concise.

Format your response EXACTLY as:
Thought: I should search the web for this.
Action: search_web
Action Input: [your search query]

After receiving an Observation, provide:
Thought: [analysis of what you found]
Result: [concise summary of findings]''',

    "chart": '''You are ChartAgent, a data visualization specialist.
You have ONE tool: plot_chart(type_title_data)
The tool input format MUST be: TYPE | Chart Title | label1=value1, label2=value2
TYPE must be one of: bar, line, pie

Your job: extract numeric data from the conversation context and create a chart.

Format your response EXACTLY as:
Thought: [identify what data to chart and the type]
Action: plot_chart
Action Input: [bar | Title | key1=val1, key2=val2]

After receiving an Observation, provide:
Thought: [confirm chart created]
Result: [description of the chart]''',
}

def run_sub_agent(agent_type, user_query, context="", max_steps=6, previous_history=None):
    agent_prompt = AGENT_PROMPTS[agent_type]
    tool_map = {
        "filings": {"search_filings": search_filings},
        "market": {"get_live_stock_price": get_live_stock_price},
        "web": {"search_web": search_web},
        "chart": {"plot_chart": plot_chart},
    }
    tools = tool_map[agent_type]
    
    system = f"{agent_prompt}\n\nContext from other agents: {context}" if context else agent_prompt
    if previous_history and len(previous_history) > 1:
        chat = []
        for i, msg in enumerate(previous_history[:-1]):
            role = msg["role"]
            content = str(msg.get("content", ""))
            if i == 0 and role == "user":
                content = f"{system}\n\n{content}"
            chat.append({"role": role, "content": content})
        chat.append({"role": "user", "content": f"User Query: {user_query}"})
    else:
        chat = [{"role": "user", "content": f"{system}\n\nUser Query: {user_query}"}]
    
    full_trace = []
    
    for step in range(max_steps):
        prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
        prompt += "Thought:"
        
        output = gemma(prompt)[0]['generated_text']
        response = "Thought:" + output[len(prompt):]
        
        # Clean up: stop at any second "Thought:" to prevent rambling
        parts = response.split("Thought:")
        if len(parts) > 2:
            response = "Thought:" + parts[1]
        
        full_trace.append({"agent": agent_type, "step": step + 1, "output": response.strip()})
        chat.append({"role": "assistant", "content": response})
        
        # Check for Result (sub-agent is done)
        if "Result:" in response:
            result_match = re.search(r"Result:\s*(.*)", response, re.DOTALL)
            result = result_match.group(1).strip() if result_match else response
            return {"result": result, "trace": full_trace}
        
        # Parse and execute tool call
        action_match = re.search(r"Action:\s*(\S+)", response)
        input_match = re.search(r"Action Input:\s*(.*)", response)
        
        if action_match and input_match:
            action = action_match.group(1).strip()
            action_input = input_match.group(1).strip().strip('"\'')
            
            if action in tools:
                observation = tools[action](action_input)
            else:
                observation = f"Error: Unknown tool {action}. Available: {list(tools.keys())}"
            
            full_trace.append({"agent": agent_type, "step": step + 1, "observation": observation})
            chat.append({"role": "user", "content": f"Observation: {observation}"})
        else:
            chat.append({"role": "user", "content": "You must use the Action/Action Input format, or provide a Result."})
    
    # If we exhausted steps, return whatever we have
    return {"result": "Agent could not complete the task in the allowed steps.", "trace": full_trace}


def run_orchestrator(user_query, previous_history=None):
    print(f"\n{'='*60}")
    print(f"QUERY: {user_query}")
    print(f"{'='*60}")
    
    # Step 1: Classify intent
    intents = classify_intent(user_query)
    print(f"Intents detected: {intents}")
    
    # Step 2: Run sub-agents
    agent_results = {}
    all_traces = []
    context = ""
    
    for intent in intents:
        print(f"\n--- Dispatching to {intent.upper()} agent ---")
        result = run_sub_agent(intent, user_query, context=context, previous_history=previous_history)
        agent_results[intent] = result["result"]
        all_traces.extend(result["trace"])
        context += f" {result['result']}"
        print(f"[{intent.upper()}] Result: {result['result']}")
    
    # Step 3: Synthesize final answer
    synthesis_prompt = f'''You are a highly intelligent financial orchestrator.
Synthesize the reports below into a comprehensive, professional response.

CRITICAL INSTRUCTIONS:
1. Format your entire response in strict Markdown (use **bold** for emphasis, lists for points).
2. Use clear spacing and newlines between paragraphs to make it highly readable.
3. If data is missing from the reports, suggest 2 or 3 follow-up QUERIES the user could ask YOU to search for instead. DO NOT suggest external actions like "contact the company" or "visit a website". The options must be actual queries the user can click to ask you.
4. You MUST output your suggested follow-ups at the very end of your response using exactly this format:
[OPTIONS: Option 1 | Option 2 | Option 3]

User asked: {user_query}

Agent Reports:
'''
    for agent_type, result in agent_results.items():
        synthesis_prompt += f"- {agent_type.title()} Agent: {result}\n"
    
    chat = [{"role": "user", "content": synthesis_prompt}]
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    output = gemma(prompt)[0]['generated_text']
    final_answer_raw = output[len(prompt):].strip()
    
    # Extract OPTIONS if present (look for bracketed text containing at least one '|')
    options = []
    options_match = re.search(r'\[([^\]]*?\|[^\]]*?)\]', final_answer_raw, re.IGNORECASE | re.DOTALL)
    if options_match:
        options_text = options_match.group(1)
        # foolproof extraction: split by "|"
        parts = [p.strip() for p in options_text.split('|')]
        for p in parts:
            clean = p.strip('"\'').strip()
            if clean: options.append(clean)
        final_answer = re.sub(r'\[([^\]]*?\|[^\]]*?)\]', '', final_answer_raw, flags=re.IGNORECASE | re.DOTALL).strip()
    else:
        final_answer = final_answer_raw
        
    print(f"\nFINAL ANSWER: {final_answer}")
    print(f"OPTIONS: {options}")
    
    # Build response for frontend
    trace_text = ""
    for t in all_traces:
        if 'output' in t:
            trace_text += f"{t['output']}\n"
        if 'observation' in t:
            trace_text += f"Observation: {t['observation']}\n"
    
    combined = trace_text + f"Final Answer: {final_answer}"
    
    return [
        {"role": "assistant", "content": combined, "agents_used": intents, "options": options}
    ]

In [ ]:
# 5. Test Run
result = run_orchestrator("What are the supply chain risks for Reliance Industries, and what is their live stock price (ticker: RELIANCE.NS)?")
for msg in result:
    print(msg["content"])
    print(msg.get("options", []))

In [ ]:
# 6. FastAPI Server + Localtunnel
import nest_asyncio
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import subprocess
import threading
import time

app = FastAPI(title="Indic-FinSight API")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

class Query(BaseModel):
    text: str
    history: list = []

@app.get("/health")
def health_check():
    return {"status": "ok", "model": loaded_model_name, "agents": ["filings", "market", "chart", "web"]}

@app.post("/chat")
def chat(query: Query):
    result = run_orchestrator(query.text, previous_history=query.history)
    return {"history": result}

def run_server():
    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=8000)

print("Starting API server...")
threading.Thread(target=run_server, daemon=True).start()
time.sleep(3)

SUBDOMAIN = "indic-finsight-seaadeep-998877"
print(f"Starting tunnel...")
lt = subprocess.Popen(["lt", "--port", "8000", "--subdomain", SUBDOMAIN], stdout=subprocess.PIPE)
print(f"\n{'='*60}")
print(f"API LIVE: https://{SUBDOMAIN}.loca.lt")
print(f"{'='*60}")

for line in lt.stdout:
    decoded = line.decode('utf-8').strip()
    if "your url is" in decoded.lower():
        print(decoded)
        break

import asyncio
async def keep_alive():
    while True:
        await asyncio.sleep(3600)

await keep_alive()